# Assignment: Video Quality Inference

To this point in the class, you have learned various techniques for leading and analyzing packet captures of various types, generating features from those packet captures, and training and evaluating models using those features.

In this assignment, you will put all of this together, using a network traffic trace to train a model to automatically infer video quality of experience from a labeled traffic trace.

## Part 1: Warmup

The first part of this assignment builds directly on the hands-on activities but extends them slightly.

### Extract Features from the Network Traffic

Load the `netflix.pcap` file, which is a packet trace that includes network traffic. 


In [18]:
from netml.pparser.parser import PCAP
from netml.utils.tool import dump_data, load_data

In [ ]:
pcap = PCAP('C:/Users/merin/ml-systems-copy/docs/notebooks/data/netflix.pcap')

In [13]:
pcap.pcap2pandas()
netflix = pcap.df

In [14]:
netflix

,datetime,dns_query,dns_resp,ip_dst,ip_dst_int,ip_src,ip_src_int,is_dns,length,mac_dst,mac_dst_int,mac_src,mac_src_int,port_dst,port_src,protocol,time,time_normed
0,2018-02-11 15:10:00,"(fonts.gstatic.com.,)",None,128.93.77.234,2.153598e+09,192.168.43.72,3.232247e+09,True,77,a0:ce:c8:0d:2b:a7,176809980013479,e4:ce:8f:01:4c:54,251575813622868,53.0,55697.0,UDP,1518358200.534682,0.000000
1,2018-02-11 15:10:00,"(fonts.gstatic.com.,)",None,128.93.77.234,2.153598e+09,192.168.43.72,3.232247e+09,True,77,a0:ce:c8:0d:2b:a7,176809980013479,e4:ce:8f:01:4c:54,251575813622868,53.0,59884.0,UDP,1518358200.534832,0.000150
2,2018-02-11 15:10:00,"(googleads.g.doubleclick.net.,)",None,128.93.77.234,2.153598e+09,192.168.43.72,3.232247e+09,True,87,a0:ce:c8:0d:2b:a7,176809980013479,e4:ce:8f:01:4c:54,251575813622868,53.0,61223.0,UDP,1518358200.539408,0.004726
3,2018-02-11 15:10:00,"(googleads.g.doubleclick.net.,)",None,128.93.77.234,2.153598e+09,192.168.43.72,3.232247e+09,True,87,a0:ce:c8:0d:2b:a7,176809980013479,e4:ce:8f:01:4c:54,251575813622868,53.0,58785.0,UDP,1518358200.541204,0.006522
4,2018-02-11 15:10:00,"(ytimg.l.google.com.,)",None,128.93.77.234,2.153598e+09,192.168.43.72,3.232247e+09,True,78,a0:ce:c8:0d:2b:a7,176809980013479,e4:ce:8f:01:4c:54,251575813622868,53.0,51938.0,UDP,1518358200.545785,0.011103
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
141466,2018-02-11 15:18:16,None,None,52.208.128.101,8.860796e+08,192.168.43.72,3.232247e+09,False,54,a0:ce:c8:0d:2b:a7,176809980013479,e4:ce:8f:01:4c:54,251575813622868,443.0,58518.0,TCP,1518358696.299107,495.764425
141467,2018-02-11 15:18:16,None,None,52.208.128.101,8.860796e+08,192.168.43.72,3.232247e+09,False,54,a0:ce:c8:0d:2b:a7,176809980013479,e4:ce:8f:01:4c:54,251575813622868,443.0,58518.0,TCP,1518358696.299132,495.764450
141468,2018-02-11 15:18:16,None,None,104.31.113.215,1.746891e+09,192.168.43.72,3.232247e+09,False,54,a0:ce:c8:0d:2b:a7,176809980013479,e4:ce:8f:01:4c:54,251575813622868,80.0,58530.0,TCP,1518358696.299136,495.764454
141469,2018-02-11 15:18:16,None,None,192.168.43.72,3.232247e+09,172.217.18.195,2.899907e+09,False,66,e4:ce:8f:01:4c:54,251575813622868,a0:ce:c8:0d:2b:a7,176809980013479,58514.0,443.0,TCP,1518358696.313540,495.778858


### Identifying the Service Type

Use the DNS traffic to filter the packet trace for Netflix traffic.

In [17]:
dns_netflix = netflix.loc[netflix["is_dns"] == True]
dns_netflix

,datetime,dns_query,dns_resp,ip_dst,ip_dst_int,ip_src,ip_src_int,is_dns,length,mac_dst,mac_dst_int,mac_src,mac_src_int,port_dst,port_src,protocol,time,time_normed
0,2018-02-11 15:10:00,"(fonts.gstatic.com.,)",None,128.93.77.234,2.153598e+09,192.168.43.72,3.232247e+09,True,77,a0:ce:c8:0d:2b:a7,176809980013479,e4:ce:8f:01:4c:54,251575813622868,53.0,55697.0,UDP,1518358200.534682,0.000000
1,2018-02-11 15:10:00,"(fonts.gstatic.com.,)",None,128.93.77.234,2.153598e+09,192.168.43.72,3.232247e+09,True,77,a0:ce:c8:0d:2b:a7,176809980013479,e4:ce:8f:01:4c:54,251575813622868,53.0,59884.0,UDP,1518358200.534832,0.000150
2,2018-02-11 15:10:00,"(googleads.g.doubleclick.net.,)",None,128.93.77.234,2.153598e+09,192.168.43.72,3.232247e+09,True,87,a0:ce:c8:0d:2b:a7,176809980013479,e4:ce:8f:01:4c:54,251575813622868,53.0,61223.0,UDP,1518358200.539408,0.004726
3,2018-02-11 15:10:00,"(googleads.g.doubleclick.net.,)",None,128.93.77.234,2.153598e+09,192.168.43.72,3.232247e+09,True,87,a0:ce:c8:0d:2b:a7,176809980013479,e4:ce:8f:01:4c:54,251575813622868,53.0,58785.0,UDP,1518358200.541204,0.006522
4,2018-02-11 15:10:00,"(ytimg.l.google.com.,)",None,128.93.77.234,2.153598e+09,192.168.43.72,3.232247e+09,True,78,a0:ce:c8:0d:2b:a7,176809980013479,e4:ce:8f:01:4c:54,251575813622868,53.0,51938.0,UDP,1518358200.545785,0.011103
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
141239,2018-02-11 15:18:12,"(plus.l.google.com.,)","(172.217.18.206,)",192.168.43.72,3.232247e+09,128.93.77.234,2.153598e+09,True,87,e4:ce:8f:01:4c:54,251575813622868,a0:ce:c8:0d:2b:a7,176809980013479,33916.0,53.0,UDP,1518358692.204593,491.669911
141245,2018-02-11 15:18:12,"(www.google.com.,)","(172.217.18.196,)",192.168.43.72,3.232247e+09,128.93.77.234,2.153598e+09,True,84,e4:ce:8f:01:4c:54,251575813622868,a0:ce:c8:0d:2b:a7,176809980013479,56870.0,53.0,UDP,1518358692.217483,491.682801
141246,2018-02-11 15:18:12,"(www.gstatic.com.,)","(216.58.209.227,)",192.168.43.72,3.232247e+09,128.93.77.234,2.153598e+09,True,85,e4:ce:8f:01:4c:54,251575813622868,a0:ce:c8:0d:2b:a7,176809980013479,34182.0,53.0,UDP,1518358692.218175,491.683493
141371,2018-02-11 15:18:13,"(freegeoip.net.,)",None,128.93.77.234,2.153598e+09,192.168.43.72,3.232247e+09,True,73,a0:ce:c8:0d:2b:a7,176809980013479,e4:ce:8f:01:4c:54,251575813622868,53.0,11404.0,UDP,1518358693.222075,492.687393


### Generate Statistics

Generate statistics and features for the Netflix traffic flows. Use the `netml` library or any other technique that you choose to generate a set of features that you think would be good features for your model. 

In [19]:
pcap.pcap2flows()

In [30]:
#Feature generation (IAT)
pcap.flow2features('IAT', fft=False, header=False)
iat_features = pcap.features

In [31]:
#Feature generation (STATS)
pcap.flow2features('STATS', fft=False, header=False)
stat_features = pcap.features

In [32]:
#Feature generation (SIZE)
pcap.flow2features('SIZE', fft=False, header=False)
size_features = pcap.features

In [33]:
size_features

array([[ 78.,  66., 200., ...,   0.,   0.,   0.],
       [ 78.,  66., 200., ...,   0.,   0.,   0.],
       [ 78.,  66.,  66., ...,   0.,   0.,   0.],
       ...,
       [ 78.,  54., 200., ...,   0.,   0.,   0.],
       [ 74.,  66., 200., ...,   0.,   0.,   0.],
       [ 66.,  60., 200., ...,   0.,   0.,   0.]])

**Write a brief justification for the features that you have chosen.**

I think that the IAT features, all the standard measurements and statistics included in STATS, and the time series size information are minimum information needed for this model. IAT and SIZE can help with measure ABR for this video application.

### Inferring Segment downloads

In addition to the features that you could generate using the `netml` library or similar, add to your feature vector a "segment downloads rate" feature, which indicates the number of video segments downloaded for a given time window.

Note: If you are using the `netml` library, generating features with `SAMP` style options may be useful, as this option gives you time windows, and you can then simply add the segment download rate to that existing dataframe.

## Part 2: Video Quality Inference

You will now load the complete video dataset from a previous study to train and test models based on these features to automatically infer the quality of a streaming video flow.

For this part of the assignment, you will need two pickle files, which we provide for you by running the code below:

```

!gdown 'https://drive.google.com/uc?id=1N-Cf4dJ3fpak_AWgO05Fopq_XPYLVqdS' -O netflix_session.pkl
!gdown 'https://drive.google.com/uc?id=1PHvEID7My6VZXZveCpQYy3lMo9RvMNTI' -O video_dataset.pkl

```

### Load the File

Load the video dataset pickle file.

### Clean the File

1. The dataset contains video resolutions that are not valid. Remove entries in the dataset that do not contain a valid video resolution. Valid resolutions are 240, 360, 480, 720, 1080.

2. The file also contains columns that are unnecessary (in fact, unhelpful!) for performing predictions. Identify those columns, and remove them.

**Briefly explain why you removed those columns.**

### Prepare Your Data

Prepare your data matrix, determine your features and labels, and perform a train-test split on your data.

### Train and Tune Your Model

1. Select a model of your choice.
2. Train the model using your training data.

### Tune Your Model

Perform hyperparameter tuning to find optimal parameters for your model.

### Evaluate Your Model

Evaluate your model accuracy according to the following metrics:

1. Accuracy
2. F1 Score
3. Confusion Matrix
4. ROC/AUC

## Part 3: Predict the Ongoing Resolution of a Real Netflix Session

Now that you have your model, it's time to put it in practice!

Use a preprocessed Netflix video session to infer **and plot** the resolution at 10-second time intervals.